# Optimización de Hiperparámetros VCP con Optuna

Busca la mejor configuración de ~12 parámetros del detector VCP usando TPE (Tree-structured Parzen Estimator).

**Prerequisitos:**
```bash
pip install optuna plotly mlflow
```

## 1. Setup e imports

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from autoresearch import (
    SwingCache,
    filter_tickers_by_start_date,
    find_common_period,
    get_ticker_info,
    load_universe,
    build_objective_function,
    create_study,
    run_backtest_for_params,
    compute_objective_score,
    study_to_dataframe,
    top_trials_summary,
    param_importance,
    reconstruct_pipeline_params,
    trades_to_dataframe,
    MLflowOptunaLogger,
    DEFAULT_RISK_PARAMS,
)

pd.set_option("display.float_format", "{:.4f}".format)
DATA_DIR = project_root / "data" / "csv"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")
print("Imports OK")

## 2. Cargar y filtrar universo de tickers

In [2]:
ticker_info = get_ticker_info(DATA_DIR)
display(ticker_info)

,start_date,end_date,n_bars
ticker,,,
AAPL,2015-01-02,2026-04-08,2832
AMZN,2015-01-02,2026-04-08,2832
AVGO,2015-01-02,2026-04-08,2832
BRK.B,2015-01-02,2026-04-10,2834
COIN,2021-04-14,2026-04-10,1254
GLD,2015-01-02,2026-04-10,2834
GOOGL,2015-01-02,2026-04-08,2832
HOOD,2021-07-29,2026-04-10,1180
IWM,2015-01-02,2026-04-10,2834


In [3]:
opt_tickers, oos_tickers = filter_tickers_by_start_date(
    DATA_DIR, max_start_date="2015-12-31", exclude_tickers=["META"]
)
print(f"Optimization set ({len(opt_tickers)}): {opt_tickers}")
print(f"Out-of-sample set ({len(oos_tickers)}): {oos_tickers}")

Optimization set (18): ['AAPL', 'AMZN', 'AVGO', 'BRK.B', 'GLD', 'GOOGL', 'IWM', 'JPM', 'MSFT', 'NVDA', 'QLD', 'QQQ', 'SLV', 'SPY', 'SQQQ', 'TIL', 'TLT', 'TQQQ']
Out-of-sample set (4): ['COIN', 'HOOD', 'PLTR', 'SOFI']


In [4]:
universe = load_universe(opt_tickers, DATA_DIR)
common_start, common_end = find_common_period(universe)
print(f"Universe: {len(universe)} tickers")
print(f"Common period: {common_start.date()} to {common_end.date()}")
print(f"Bars per ticker: {[len(df) for df in universe.values()]}")

Universe: 18 tickers
Common period: 2015-05-27 to 2026-04-08
Bars per ticker: [2832, 2832, 2832, 2834, 2834, 2832, 2834, 2834, 2832, 2832, 2832, 2832, 2834, 2832, 2832, 1814, 2832, 2832]


## 3. Baseline: parámetros del experimento original

Corremos el pipeline con los parámetros conocidos (`volume_ratio_threshold=1.5`) como referencia.

In [5]:
from models.configs import ATRZigZagConfig

BASELINE_PARAMS = {
    "swing_config": ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False),
    "sequence_params": {
        "method": "tolerance",
        "min_contractions": 2,
        "max_contractions": 6,
        "lookback_bars": 126,
        "tolerance": 0.10,
        "max_depth_pct": 0.35,
        "min_total_reduction": 0.80,
        "max_gap_between_contractions_days": None,
    },
    "compression_params": {
        "method": "ratio",
        "atr_period": 14,
        "ratio_threshold": 0.85,
    },
    "volume_contraction_params": {
        "method": "ratio",
        "volume_column": "volume",
        "ratio_threshold": 0.85,
    },
    "breakout_params": {
        "volume_method": "ratio",
        "volume_ratio_threshold": 1.5,
        "volume_lookback_days": 50,
        "require_volume_confirmation": True,
    },
    "grouping_params": {"max_gap_days": 30},
    "risk_params": dict(DEFAULT_RISK_PARAMS),
}

In [6]:
baseline_result = run_backtest_for_params(universe, BASELINE_PARAMS)
baseline_metrics = baseline_result["metrics"]
baseline_score = compute_objective_score(baseline_metrics)

print("=== BASELINE ===")
print(f"  n_trades:       {baseline_metrics['n_trades']}")
print(f"  expectancy_r:   {baseline_metrics['expectancy_r']:+.4f}")
print(f"  win_rate:       {baseline_metrics['win_rate']:.1%}")
print(f"  profit_factor:  {baseline_metrics['profit_factor']:.2f}")
print(f"  score (Opt C):  {baseline_score:+.4f}")
print(f"\n  Trades per ticker:")
for ticker, n in sorted(baseline_metrics["trades_per_ticker"].items()):
    print(f"    {ticker}: {n}")

=== BASELINE ===
  n_trades:       78
  expectancy_r:   +0.3058
  win_rate:       46.2%
  profit_factor:  1.62
  score (Opt C):  +2.7004

  Trades per ticker:
    AAPL: 6
    AMZN: 14
    AVGO: 6
    BRK.B: 3
    GLD: 4
    GOOGL: 10
    IWM: 2
    JPM: 6
    MSFT: 5
    NVDA: 2
    QLD: 2
    QQQ: 2
    SLV: 4
    SPY: 4
    SQQQ: 2
    TIL: 1
    TLT: 3
    TQQQ: 2


## 4. Configurar y correr optimización (100 trials)

TPESampler con `multivariate=True`, 20 startup trials random, seed=42.
Cada trial corre el pipeline completo sobre los 18 tickers. SwingCache
evita recomputar pasos 1-2 cuando el swing_config se repite entre trials.

In [ ]:
N_TRIALS = 100
STUDY_NAME = "vcp_optuna_phase1"
MLFLOW_EXPERIMENT = "autoresearch_vcp_phase1"

In [ ]:
cache = SwingCache()
objective = build_objective_function(universe, cache=cache)
study = create_study(study_name=STUDY_NAME, n_startup_trials=20, seed=42)
logger = MLflowOptunaLogger(experiment_name=MLFLOW_EXPERIMENT)

with logger.parent_run(study_name=STUDY_NAME, tags={"n_tickers": str(len(universe))}):
    study.optimize(objective, n_trials=N_TRIALS, callbacks=[logger.optuna_callback])
    logger.log_study_summary(study, n_tickers=len(universe))

print(f"\nOptimización completada: {len(study.trials)} trials")
print(f"Best score: {study.best_value:+.4f} (trial #{study.best_trial.number})")
print(f"Cache stats: {cache.stats()}")

## 5. Inspeccionar resultados

### 5a. Best trial

In [9]:
best = study.best_trial
print(f"Best trial: #{best.number}")
print(f"  score:         {best.value:+.4f}")
print(f"  n_trades:      {best.user_attrs['n_trades']}")
print(f"  expectancy_r:  {best.user_attrs['expectancy_r']:+.4f}")
print(f"  win_rate:      {best.user_attrs['win_rate']:.1%}")
print(f"  profit_factor: {best.user_attrs['profit_factor']:.2f}")
print(f"\nParams:")
for k, v in sorted(best.params.items()):
    print(f"  {k}: {v}")

Best trial: #6
  score:         +3.3287
  n_trades:      43
  expectancy_r:  +0.5076
  win_rate:      44.2%
  profit_factor: 2.56

Params:
  atr_length: 10
  atr_mult: 3.25
  compression_threshold: 0.9157758564688984
  lookback_bars: 130
  max_contractions: 7
  max_depth_pct: 0.32169314570885454
  max_gap_days: 21
  min_contractions: 3
  min_total_reduction: 0.6789672648812825
  tolerance: 0.05
  vol_contraction_threshold: 0.8746596253655116
  volume_ratio_threshold: 1.5316286173968545


### 5b. Top 10 trials

In [10]:
top10 = top_trials_summary(study, n=10)
display(top10)

,trial_number,score,n_trades,expectancy_r,win_rate,profit_factor,atr_length,atr_mult,min_contractions,max_contractions,lookback_bars,tolerance,max_depth_pct,min_total_reduction,compression_threshold,vol_contraction_threshold,volume_ratio_threshold,max_gap_days
0,6,3.3287,43,0.5076,0.4419,2.5648,10,3.2500,3,7,130,0.0500,0.3217,0.6790,0.9158,0.8747,1.5316,21
1,0,3.0531,16,0.7633,0.4375,3.8732,15,3.5000,3,6,90,0.0750,0.2616,0.8665,0.8503,0.8916,1.3144,40
2,1,2.7230,66,0.3352,0.4545,1.6725,23,1.7500,2,5,100,0.1250,0.3364,0.7228,0.8530,0.7779,1.5045,27
3,15,2.3071,65,0.2862,0.4000,1.6142,15,1.7500,3,7,90,0.1500,0.4134,0.7888,0.8324,0.7984,1.3652,38
4,5,2.0657,16,0.5164,0.4375,2.0882,16,2.0000,3,6,90,0.1250,0.2782,0.8505,0.7186,0.9474,1.8406,24
5,17,1.9246,39,0.3082,0.4872,1.6276,19,1.5000,2,6,80,0.0750,0.3597,0.8230,0.8630,0.7949,1.7985,24
6,16,1.7826,66,0.2194,0.4848,1.5872,24,2.7500,2,6,130,0.2000,0.4274,0.8450,0.8605,0.7668,1.4131,38
7,3,1.6555,21,0.3613,0.4286,1.5975,14,1.5000,3,6,80,0.1250,0.2569,0.8773,0.7647,0.8825,1.5182,30
8,13,1.6381,29,0.3042,0.3448,1.6853,20,3.0000,2,7,100,0.1500,0.3767,0.7839,0.7226,0.9171,1.5245,23
9,9,1.4690,36,0.2448,0.4444,1.4740,13,1.5000,2,5,140,0.1750,0.3767,0.8679,0.9009,0.7873,1.9248,31


### 5c. Importancia de parámetros (fANOVA)

In [11]:
imp = param_importance(study)
if imp is not None:
    display(imp)
else:
    print("param_importance no pudo calcularse (pocos trials o error de fANOVA)")

,param,importance
0,volume_ratio_threshold,0.3181
1,min_total_reduction,0.2286
2,compression_threshold,0.1301
3,max_depth_pct,0.0865
4,atr_mult,0.0724
5,atr_length,0.0715
6,max_gap_days,0.0245
7,vol_contraction_threshold,0.0242
8,lookback_bars,0.0175
9,tolerance,0.0174


### 5d. Visualizaciones de Optuna

In [12]:
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_parallel_coordinate,
        plot_param_importances,
        plot_slice,
    )
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly no instalado. Correr: pip install plotly")

In [13]:
if HAS_PLOTLY:
    fig = plot_optimization_history(study)
    fig.update_layout(title="Optimization History", height=400)
    fig.show()

In [14]:
if HAS_PLOTLY:
    fig = plot_parallel_coordinate(
        study,
        params=[
            "atr_mult", "tolerance", "max_depth_pct",
            "compression_threshold", "vol_contraction_threshold",
            "volume_ratio_threshold",
        ],
    )
    fig.update_layout(title="Parallel Coordinate (top params)", height=500)
    fig.show()

In [15]:
if HAS_PLOTLY:
    try:
        fig = plot_param_importances(study)
        fig.update_layout(title="Parameter Importances (fANOVA)", height=400)
        fig.show()
    except Exception as e:
        print(f"plot_param_importances falló: {e}")

In [16]:
if HAS_PLOTLY:
    fig = plot_slice(
        study,
        params=["atr_mult", "tolerance", "compression_threshold", "volume_ratio_threshold"],
    )
    fig.update_layout(title="Slice Plot (key params vs score)", height=400)
    fig.show()

## 6. Re-ejecutar best trial para validar reproducibilidad

In [17]:
best_params = reconstruct_pipeline_params(study.best_trial.params)
rerun_result = run_backtest_for_params(universe, best_params)
rerun_metrics = rerun_result["metrics"]
rerun_score = compute_objective_score(rerun_metrics)

original_score = study.best_value
score_diff = abs(rerun_score - original_score)

print(f"Score original:    {original_score:+.6f}")
print(f"Score re-run:      {rerun_score:+.6f}")
print(f"Diferencia:        {score_diff:.2e}")

if score_diff > 1e-6:
    print("\n⚠ WARNING: scores difieren. Posible floating point o non-determinism.")
    print(f"  Original n_trades: {study.best_trial.user_attrs['n_trades']}")
    print(f"  Re-run n_trades:   {rerun_metrics['n_trades']}")
else:
    print("\nReproducibilidad OK: scores idénticos.")

Score original:    +3.328696
Score re-run:      +3.328696
Diferencia:        0.00e+00

Reproducibilidad OK: scores idénticos.


In [18]:
best_trades_df = trades_to_dataframe(rerun_result["all_trades"])
print(f"Trades del best trial: {len(best_trades_df)}")
display(best_trades_df)

Trades del best trial: 43


,trade_num,ticker,entry_date,exit_date,exit_reason,duration_days,entry_price,exit_price,pnl_pct,r_multiple,max_r,n_contractions,stop_method,stop_distance_pct
0,1,AAPL,2016-07-27,2016-09-08,distribution,43,25.7375,26.3800,0.0250,0.3936,1.0000,4,pattern,0.0634
1,2,AAPL,2016-09-08,2016-09-09,distribution,1,26.3800,25.7825,-0.0226,-0.3236,0.0000,3,fixed_pct,0.0700
2,3,AAPL,2024-12-20,2025-01-13,stop_loss,24,254.4900,234.4000,-0.0789,-1.1277,0.2543,3,fixed_pct,0.0700
3,4,AMZN,2017-04-04,2017-06-09,distribution,66,45.3415,48.9155,0.0788,1.1261,1.6464,3,fixed_pct,0.0700
4,5,AMZN,2017-10-27,2017-12-04,distribution,38,55.0475,56.6975,0.0300,0.4282,1.2311,3,fixed_pct,0.0700
5,6,AMZN,2017-11-27,2017-12-04,distribution,7,59.7915,56.6975,-0.0517,-0.7392,0.0000,3,fixed_pct,0.0700
6,7,AVGO,2016-06-03,2016-06-24,stop_loss,21,16.2560,14.8720,-0.0851,-1.2163,0.2004,3,fixed_pct,0.0700
7,8,AVGO,2021-09-03,2021-09-20,distribution,17,49.7680,49.4820,-0.0057,-0.0821,0.3462,3,fixed_pct,0.0700
8,9,AVGO,2023-03-03,2023-05-02,distribution,60,63.2760,61.2340,-0.0323,-0.4610,0.2472,3,fixed_pct,0.0700
9,10,AVGO,2024-12-13,2025-01-27,stop_loss,45,224.8000,202.1300,-0.1008,-1.4406,1.6014,3,fixed_pct,0.0700


## 7. Tabla comparativa: baseline vs best Optuna

In [19]:
comparison = pd.DataFrame({
    "Baseline (original)": {
        "score": baseline_score,
        "n_trades": baseline_metrics["n_trades"],
        "expectancy_r": baseline_metrics["expectancy_r"],
        "win_rate": baseline_metrics["win_rate"],
        "profit_factor": baseline_metrics["profit_factor"],
        "avg_winner_r": baseline_metrics["avg_winner_r"],
        "avg_loser_r": baseline_metrics["avg_loser_r"],
    },
    "Best Optuna": {
        "score": rerun_score,
        "n_trades": rerun_metrics["n_trades"],
        "expectancy_r": rerun_metrics["expectancy_r"],
        "win_rate": rerun_metrics["win_rate"],
        "profit_factor": rerun_metrics["profit_factor"],
        "avg_winner_r": rerun_metrics["avg_winner_r"],
        "avg_loser_r": rerun_metrics["avg_loser_r"],
    },
}).T

display(comparison)

,score,n_trades,expectancy_r,win_rate,profit_factor,avg_winner_r,avg_loser_r
Baseline (original),2.7004,78.0000,0.3058,0.4615,1.6237,1.7247,-0.9105
Best Optuna,3.3287,43.0000,0.5076,0.4419,2.5648,1.8830,-0.5812


In [20]:
print("=== Parámetros del baseline vs best Optuna ===")
print(f"{'Parámetro':<30} {'Baseline':>12} {'Optuna':>12}")
print("-" * 56)

baseline_flat = {
    "atr_length": 14, "atr_mult": 2.0,
    "min_contractions": 2, "max_contractions": 6,
    "lookback_bars": 126, "tolerance": 0.10,
    "max_depth_pct": 0.35, "min_total_reduction": 0.80,
    "compression_threshold": 0.85, "vol_contraction_threshold": 0.85,
    "volume_ratio_threshold": 1.5, "max_gap_days": 30,
}
optuna_flat = study.best_trial.params

for k in baseline_flat:
    bval = baseline_flat[k]
    oval = optuna_flat.get(k, "N/A")
    if isinstance(bval, float):
        print(f"{k:<30} {bval:>12.4f} {oval:>12.4f}")
    else:
        print(f"{k:<30} {bval:>12} {oval:>12}")

=== Parámetros del baseline vs best Optuna ===
Parámetro                          Baseline       Optuna
--------------------------------------------------------
atr_length                               14           10
atr_mult                             2.0000       3.2500
min_contractions                          2            3
max_contractions                          6            7
lookback_bars                           126          130
tolerance                            0.1000       0.0500
max_depth_pct                        0.3500       0.3217
min_total_reduction                  0.8000       0.6790
compression_threshold                0.8500       0.9158
vol_contraction_threshold            0.8500       0.8747
volume_ratio_threshold               1.5000       1.5316
max_gap_days                             30           21


## 8. Distribución de trades por ticker (best trial)

In [21]:
ticker_comparison = []
for ticker in sorted(universe.keys()):
    bl_n = baseline_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    opt_n = rerun_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    ticker_comparison.append({"ticker": ticker, "baseline_trades": bl_n, "optuna_trades": opt_n})

tc_df = pd.DataFrame(ticker_comparison).set_index("ticker")
tc_df["diff"] = tc_df["optuna_trades"] - tc_df["baseline_trades"]
display(tc_df)
print(f"\nBaseline total: {tc_df['baseline_trades'].sum()}")
print(f"Optuna total:   {tc_df['optuna_trades'].sum()}")

,baseline_trades,optuna_trades,diff
ticker,,,
AAPL,6,3,-3
AMZN,14,3,-11
AVGO,6,5,-1
BRK.B,3,5,2
GLD,4,2,-2
GOOGL,10,1,-9
IWM,2,1,-1
JPM,6,3,-3
MSFT,5,2,-3



Baseline total: 78
Optuna total:   43


## 9. Todos los trials (DataFrame completo)

In [22]:
all_trials_df = study_to_dataframe(study)
print(f"Total trials: {len(all_trials_df)}")
print(f"Trials con score > 0: {(all_trials_df['score'] > 0).sum()}")
print(f"Trials con n_trades == 0: {(all_trials_df['n_trades'] == 0).sum()}")
print(f"Trials con n_trades >= 10: {(all_trials_df['n_trades'] >= 10).sum()}")
display(all_trials_df.head(20))

Total trials: 20
Trials con score > 0: 18
Trials con n_trades == 0: 0
Trials con n_trades >= 10: 17


,trial_number,score,state,atr_length,atr_mult,min_contractions,max_contractions,lookback_bars,tolerance,max_depth_pct,...,vol_contraction_threshold,volume_ratio_threshold,max_gap_days,n_trades,expectancy_r,win_rate,profit_factor,avg_winner_r,avg_loser_r,trades_per_ticker
0,6,3.3287,COMPLETE,10,3.2500,3,7,130,0.0500,0.3217,...,0.8747,1.5316,21,43,0.5076,0.4419,2.5648,1.8830,-0.5812,"{'AAPL': 3, 'AMZN': 3, 'AVGO': 5, 'BRK.B': 5, ..."
1,0,3.0531,COMPLETE,15,3.5000,3,6,90,0.0750,0.2616,...,0.8916,1.3144,40,16,0.7633,0.4375,3.8732,2.3518,-0.4723,"{'AAPL': 2, 'AMZN': 1, 'BRK.B': 3, 'GLD': 2, '..."
2,1,2.7230,COMPLETE,23,1.7500,2,5,100,0.1250,0.3364,...,0.7779,1.5045,27,66,0.3352,0.4545,1.6725,1.8339,-0.9138,"{'AAPL': 6, 'AMZN': 10, 'AVGO': 5, 'BRK.B': 3,..."
3,15,2.3071,COMPLETE,15,1.7500,3,7,90,0.1500,0.4134,...,0.7984,1.3652,38,65,0.2862,0.4000,1.6142,1.8802,-0.7766,"{'AAPL': 6, 'AMZN': 5, 'AVGO': 4, 'BRK.B': 2, ..."
4,5,2.0657,COMPLETE,16,2.0000,3,6,90,0.1250,0.2782,...,0.9474,1.8406,24,16,0.5164,0.4375,2.0882,2.2652,-0.8437,"{'AAPL': 2, 'AMZN': 2, 'GLD': 1, 'GOOGL': 2, '..."
5,17,1.9246,COMPLETE,19,1.5000,2,6,80,0.0750,0.3597,...,0.7949,1.7985,24,39,0.3082,0.4872,1.6276,1.6405,-0.9575,"{'AAPL': 3, 'AMZN': 5, 'AVGO': 4, 'BRK.B': 1, ..."
6,16,1.7826,COMPLETE,24,2.7500,2,6,130,0.2000,0.4274,...,0.7668,1.4131,38,66,0.2194,0.4848,1.5872,1.2233,-0.7254,"{'AAPL': 2, 'AMZN': 12, 'AVGO': 5, 'BRK.B': 4,..."
7,3,1.6555,COMPLETE,14,1.5000,3,6,80,0.1250,0.2569,...,0.8825,1.5182,30,21,0.3613,0.4286,1.5975,2.2538,-1.0581,"{'AAPL': 2, 'AMZN': 3, 'AVGO': 2, 'BRK.B': 2, ..."
8,13,1.6381,COMPLETE,20,3.0000,2,7,100,0.1500,0.3767,...,0.9171,1.5245,23,29,0.3042,0.3448,1.6853,2.1695,-0.6775,"{'AAPL': 1, 'AMZN': 4, 'AVGO': 1, 'BRK.B': 1, ..."
9,9,1.4690,COMPLETE,13,1.5000,2,5,140,0.1750,0.3767,...,0.7873,1.9248,31,36,0.2448,0.4444,1.4740,1.7130,-0.9297,"{'AAPL': 2, 'AMZN': 5, 'AVGO': 6, 'BRK.B': 1, ..."


## 10. Conclusiones y próximos pasos

### Resultados
- **Baseline score** vs **Best Optuna score**: ver tabla comparativa arriba.
- Los parámetros que más impactan el score se ven en la tabla de fANOVA importance.
- La distribución de trades por ticker muestra si Optuna concentra señales en pocos tickers o diversifica.

### Limitaciones de esta Fase 1
- **In-sample optimization**: estamos optimizando y evaluando sobre el mismo periodo. No hay validación out-of-sample.
- **Sin walk-forward**: no probamos si los parámetros generalizan a periodos futuros.
- **Solo `method=tolerance`**: no exploramos `robust_trend` ni otros métodos de compresión.
- **50 trials**: con 12 parámetros, TPE necesita más trials para convergir bien (~200-500).

### Próximos pasos (Fase 2)
1. **Walk-forward validation**: dividir el periodo en ventanas train/test y optimizar rolling.
2. **Out-of-sample test**: correr best params sobre los tickers excluidos (COIN, HOOD, PLTR, SOFI).
3. **Más trials**: escalar a 200+ trials para explorar mejor el espacio.
4. **Explorar métodos**: agregar `robust_trend` y `ratio_normalized` al search space.
5. **Multi-objective**: optimizar expectancy y n_trades como objetivos separados (Pareto front).

### MLflow
Para explorar los resultados en detalle:
```bash
cd /home/gdelarosa/proyectos/deteccion-vcp && mlflow ui --backend-store-uri mlruns/
```